## **Step 1: At the start of every session**

#### Cell A: Drive, wandb, and the GPU

In [1]:
from google.colab import drive, userdata
import json, os, pathlib, subprocess
drive.mount("/content/drive")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
DRIVE = pathlib.Path("/content/drive/MyDrive/msml612")
DATASET = DRIVE / "physionet_set_a.npz"
!nvidia-smi --query-gpu=name --format=csv,noheader

Mounted at /content/drive
NVIDIA A100-SXM4-80GB


This cell must print NVIDIA A100-SXM4-80GB. If it prints anything else, disconnect and try again later. Never use another GPU.

#### Cell B: the code, pinned to the dry run's commit

In [5]:
!git clone -q https://github.com/jkelle11-source/msml612_project.git
%cd /content/msml612_project
started = DRIVE / "dry-run" / "sweep.json"
if started.exists():        # a later session: the commit the dry run started from
    subprocess.run(["git", "checkout", "-q", json.loads(started.read_text())["commit"]], check=True)
!git log --oneline -1
!uv sync --locked
!uv run python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"
!sha256sum {DATASET}

/content/msml612_project
0da5b38 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #57 from jkelle11-source/m6-2_jake
Resolved 67 packages in 1ms
Checked 63 packages in 0.73ms
2.14.1+cu130 13.0 True
ce6ec696eee3440a08ff6cd81b311b2a4050487c396dfed66e93edc7d7458070  /content/drive/MyDrive/msml612/physionet_set_a.npz


Check three things:
- The commit is `Merge pull request #57`, or the same commit in every later session.
- The torch line reads `2.14.1+cu130 13.0 True`.
- The hash is `ce6ec696…7458070`.

## **Step 2: The dry run, about 16.4h**

#### Cell C

In [6]:
!uv run python -m diffusion.sweep --tag dry-run --epochs 20 --seeds 42 43 --out {DRIVE}/dry-run --dataset {DATASET}

12 cells on cuda -> /content/drive/MyDrive/msml612/dry-run
[1/12] base_random_seed42
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: jakekeller864 (jkelle11-university-of-maryland) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run wtdxtr3c (0.0s)
wandb: ⣻ setting up run wtdxtr3c (0.0s)
wandb: ⣽ setting up run wtdxtr3c (0.0s)
wandb: ⣾ setting up run wtdxtr3c (0.0s)
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in /content/msml612_project/wandb/run-20261117_055027-wtdxtr3c
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run base_random_seed42
wandb: ⭐️ View project at https://wandb.ai/DATA612/msml612-csdi
wandb: 🚀 View run at https://wandb.ai/DATA612/msml612-csdi/runs/wtdxtr3c
/content/msml612_project/models/csdi/diff_models.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_lay

- **First cell as a smoke test.** The first line is `12 cells on cuda -> ...`, then `[1/12] base_random_seed42`. That cell takes about 80 min: about 2.5 min of training, then two 39-min scoring passes printing `batch i/50 running MAE ...`. Once its first scoring finishes, wandb should show `base_random_seed42` tagged `dry-run`, with GPU `NVIDIA A100-SXM4-80GB` and the `random/0.1/test/*` keys.
- **Cell order:** random-regime seeds 42 then 43 (base, ext1, ext2, ext1+2), then block-aware seeds 42 and 43 (base, ext1).
- **If a session is lost:** start a new one and run Cells A, B, and C again. Finished cells print `trained` and `scored on the ... holdout`. An unfinished run either resumes (`resuming ... from its checkpoint`) or starts again.
- **To end a session yourself:** run `drive.flush_and_unmount()` first.
- **It's done** when it prints `done: collect and check them with ... diffusion.results --tag dry-run`.

## **Step 3: The rerun check, about 1.4h**

Run this after the dry run, in the same session or a new one after Cells A and B.

#### Cell D

In [7]:
!uv run python -m diffusion.sweep --tag rerun --epochs 20 --regime random --variants base --seeds 42 --out {DRIVE}/rerun --dataset {DATASET}
!uv run python -m diffusion.sweep --regime random --variants base --seeds 42 --out {DRIVE}/dry-run --compare {DRIVE}/rerun

1 cells on cuda -> /content/drive/MyDrive/msml612/rerun
[1/1] base_random_seed42
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: jakekeller864 (DATA612) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run 2bjwiv95 (0.0s)
wandb: ⣻ setting up run 2bjwiv95 (0.0s)
wandb: ⣽ setting up run 2bjwiv95 (0.0s)
wandb: ⣾ setting up run 2bjwiv95 (0.0s)
wandb: ⣷ setting up run 2bjwiv95 (0.5s)
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in /content/msml612_project/wandb/run-20261117_222230-2bjwiv95
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run base_random_seed42
wandb: ⭐️ View project at https://wandb.ai/DATA612/msml612-csdi
wandb: 🚀 View run at https://wandb.ai/DATA612/msml612-csdi/runs/2bjwiv95
/content/msml612_project/models/csdi/diff_models.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because

The second line should print `base_random_seed42: the same numbers`. Because Cell B pins to the dry run's commit, the rerun uses the same code.